# Анализ закупок группы Сбер за 2024-2025 годы

Notebook построен из текущего clean source sprint merge  без старых curated snapshots.

- Batch-и: `ast-full-2024-2025-finalcheck  b2b_center_prompt2_full_scope_2026-06-18  eis-prompt2-full-scope-2026-06-22`
- Строк до cross-source dedupe: `3164`
- Удалено cross-source дублей: `3`
- Уникальных закупок после dedupe: `3161`
- Строк с раскрытой ценой: `900`
- Сумма раскрытых цен: `30 475 672 417.80` RUB

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MERGED = ROOT / 'output' / 'merged_sprints.csv'
SUMMARY = ROOT / 'output' / 'merged_sprints_summary.json'
lots = pd.read_csv(MERGED, dtype=str, keep_default_na=False)
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
lots['price_rub_num'] = pd.to_numeric(lots.get('price_rub', ''), errors='coerce')
lots.head()

## Источники

В clean dataset входят только источники из `configs/source_sprints_allowlist.csv`.

In [ ]:
source_counts = lots['source_system'].value_counts()
display(source_counts.rename_axis('source_system').reset_index(name='rows'))
source_counts.plot(kind='bar', figsize=(8, 4), color='#2A9D8F')
plt.title('Rows by source')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

## Дедупликация

Основной ключ: `procedure_number + lot_number`. Между источниками используется тот же ключ без `source_system`, поэтому EIS-control строки удаляются как дубли уже найденных процедур.

In [ ]:
pd.DataFrame([summary])

## Активность по годам

Дата берётся из `published_at`, где источник её отдаёт.

In [ ]:
dates = pd.to_datetime(lots.get('published_at', ''), errors='coerce')
yearly = lots.assign(publication_year=dates.dt.year)
yearly = yearly[yearly['publication_year'].isin([2024, 2025])]
yearly_summary = yearly.groupby('publication_year').agg(
    rows=('procedure_number', 'count'),
    priced_rows=('price_rub_num', lambda s: int(s.notna().sum())),
    disclosed_price_rub=('price_rub_num', 'sum'),
).reset_index()
display(yearly_summary)
yearly_summary.plot(x='publication_year', y='rows', kind='bar', legend=False, figsize=(7, 4), color='#457B9D')
plt.title('Rows by publication year')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

## Топ заказчиков/юрлиц

Этот блок показывает, какие entity дают основной наблюдаемый объём.

In [ ]:
entity_counts = lots['entity_name'].value_counts().head(20)
display(entity_counts.rename_axis('entity_name').reset_index(name='rows'))
entity_counts.sort_values().plot(kind='barh', figsize=(10, 7), color='#E76F51')
plt.title('Top entities by rows')
plt.xlabel('Rows')
plt.tight_layout()
plt.show()

## Топ лотов с раскрытой ценой

Цена раскрыта не во всех источниках, поэтому список является shortlist для ручной проверки, а не полным рейтингом расходов.

In [ ]:
top_priced = lots[lots['price_rub_num'].notna()].sort_values('price_rub_num', ascending=False)
cols = [c for c in ['source_system', 'entity_name', 'procedure_number', 'lot_number', 'subject', 'price_rub_num', 'published_at', 'detail_url'] if c in top_priced.columns]
display(top_priced[cols].head(20))

## Вывод

Проект сейчас находится в clean source sprint состоянии: старые probe/diag/generated артефакты удалены, итоговая статистика воспроизводится через `scripts/merge_sprints.py`, а правила включения новых источников закреплены в документации и тестах.